In [ ]:
import torch
import os
from datasets import load_from_disk
from utils import get_importance_score, get_dataloader
import plotly.graph_objects as go

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-128k-instruct", use_fast=False)
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="auto",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
) 

# Code

In [ ]:
__DIR__ = "../data/preprocessed/"

In [ ]:
dataset_folders = [os.path.join(__DIR__, f) for f in os.listdir(__DIR__)]
print(dataset_folders)
dataset = load_from_disk(dataset_folders[0])

In [ ]:
dataloader = get_dataloader(dataset, tokenizer, batch_size=1, max_length=4096)

In [ ]:
target_layers = list(range(16,32))

In [ ]:
head_importance = get_importance_score(model, dataloader, target_layers)

In [ ]:

heatmap = go.Heatmap(z=head_importance[target_layers,:].cpu().numpy())
fig = go.Figure(data=heatmap)
fig.update_layout(
    xaxis_title="Attention Heads",
    yaxis_title="Layers",
    title=f"Dataset {dataset_folders[0].split('/')[-1]}",
)
fig.show()

In [ ]:
for ds in dataset_folders:
    dataset = load_from_disk(ds)
    dataloader = get_dataloader(dataset, tokenizer, batch_size=1, max_length=4096)
    head_importance = get_importance_score(model, dataloader, target_layers)
    
    heatmap = go.Heatmap(z=head_importance[target_layers,:].cpu().numpy())
    fig = go.Figure(data=heatmap)
    fig.update_layout(
        xaxis_title="Attention Heads",
        yaxis_title="Layers",
        title=f"Dataset {ds.split('/')[-1]}",
    )
    fig.show()